In [1]:
import arcpy
import os
import time
import re

In [2]:
# Uncheck the World Topographic Map and World Hillshade in content pane before running the script.
# Update the path and Utility base upon data.

utility = "UNION"
map_type = "LANDBASE"
package_dir = r"C:\Users\sameer.bajracharya\Sam_works\DEV\ArcGIS_Enterprise\VTPK\VTPK_"
wkid_to_set = 3857 # The WKID for WGS_1984_Web_Mercator_Auxiliary_Sphere is 3857

# Update scale for all the layers
# Landbase scale 1:1000000 - 1
# Network scale 1:1000000 - 50000

default_min_scale = 100000
default_max_scale = 1

label_min_scale_30000= 50000
label_max_scale_30000 = 1 

label_min_scale_10000= 10000
label_max_scale_10000 = 1

layers_to_skip = [
    "World Topographic Map",
    "World Hillshade",
    "Basemap"
]

layers_to_30000 = [
    "CO",
    "NODE_LEADER_LINE",
    "MAIN_SECONDARY_DIMENSION",
    "MAIN_PRIMARY_DIMENSION",
    "DESIGN_MAINS",
    "NOTIFICATION",
    "CASING"
]

layers_to_10000 = [
    "POLE",
    "HUBS",
    "MANHOLES",
    "POP",
    "VAULT",
    "HANDHOLES",
    "PULL_BOX",
    "NATIONAL_FOTS",
    "DESIGN_REG_STATION",
    "DESIGN_FITTINGS",
    "PUP"
]
print(f"Parameter set for {utility} {map_type} with {wkid_to_set} coordinate")


Parameter set for BELLONTARIO NETWORK with 3857 coordinate


In [31]:
# Set the custom scales for the layers

def set_layer_scale(layer_list):
    target_text = "_text" # for updating the text label with 10000 scale.
    if layer_list is None:
        return

    for layer in layer_list:
        try:
            # Check if the layer's name is skip list
            if layer.name in layers_to_skip:
                print(f"\nSkipping Base Maps: {layer.name}")
                continue  # This moves to the next layer in the loop
            
            if layer.supports("minThreshold") and layer.supports("maxThreshold"):
                # print(f"Setting scale for: {layer.name}")
                # layer.minThreshold = default_min_scale
                # layer.maxThreshold = default_max_scale
                # if target_text.lower() in layer.name.lower():
                #     print(f"Setting scale for: {layer.name} with {label_min_scale_10000}")
                #     layer.minThreshold = label_min_scale_10000
                #     layer.maxThreshold = label_max_scale_10000
                    
                if layer.name in layers_to_30000:
                    print(f"Setting scale for: {layer.name} with {label_min_scale_30000}")
                    layer.minThreshold = label_min_scale_30000
                    layer.maxThreshold = label_max_scale_30000
                    
                elif layer.name in layers_to_10000:
                    print(f"Setting scale for: {layer.name} with {label_min_scale_10000}")
                    layer.minThreshold = label_min_scale_10000
                    layer.maxThreshold = label_max_scale_10000
                    
                else:
                    print(f"Setting scale for: {layer.name} with {default_min_scale}")
                    layer.minThreshold = default_min_scale
                    layer.maxThreshold =default_max_scale
            else:
                print(f"\nSkipping (not supported): {layer.name}")

            # If the layer is a group layer, recursively call this function
            # if layer.isGroupLayer:
            #     print(f"\n--- Entering Group: {layer.name} ---\n")
            #     set_layer_scale(layer.listLayers())
                # print(f"--- Exiting Group: {layer.name} ---")

        except Exception as e:
            print(f"Error setting scale for {layer.name}: {e}")
            
print("Completed running 'Set_Layer_scale function'")

Completed running 'Set_Layer_scale function'


In [32]:
# Finds all unique codes in the layer names within a given map
   
def extract_layer_codes(name_list):
    """
    Args:
        map_names(Name of the map consisting of codes e.g New_data_ON01)

    Returns:
        set: A set of all unique codes found.
    """
    results = []
    
    # Loop through each item in the input list
    for text in name_list:
        # Use re.split to break the string apart by the underscore '_'
        parts = re.split(r"_", text) 
        
        # Get the last element of the resulting list
        last_segment = parts[-1] 
        
        # Add the result to the output list
        results.append(last_segment)
        
    return results
       
print("Completed running function 'Extract Layer Codes'")

Completed running function 'Extract Layer Codes'


In [33]:
# Get the current ArcGIS Pro project and the active map.
try:
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    active_map = aprx.activeMap
        
    print(f"The active map is: {active_map.name}\n")

    # Set the path to your project's geodatabase for the output.
    # workspace_gdb = r"C:\path\to\your\project.gdb"
    # maps = aprx.listMaps()

    if not active_map:
        print(f"No Maps found in the project.")
    else:
        # Creating a SpatialReference object from the WKID
        new_cs = arcpy.SpatialReference(wkid_to_set)

        # Set the map's spatial reference
        active_map.spatialReference = new_cs

        print(f"Successfully set map CS to: {active_map.spatialReference.name}\n")
        
        # Get the list of layers from the map
        set_layer_scale(active_map.listLayers()) # Enable to set the scale for individual sublayers
        print(f"\n ----- Scale Updated.-----")        

     # Get the layers from the map
    all_layers_list = active_map.listLayers()
    
    if not all_layers_list:
        print("Error: No layers found in the active map.")
    else:
        # Flag to track if we find a valid layer
        found_valid_layer = False 
        
        first_layer = [all_layers_list[0].name]
        print(f"The first Layer in TOC is : {first_layer}")

        map_names = extract_layer_codes(first_layer)
        
        old_map_name = active_map.name
        new_map_name = list(map_names)[0]

        print(f"\nRenaming active map from '{old_map_name}' to '{new_map_name}'...")
        active_map.name = new_map_name

        map_view = aprx.activeView

        thumb_loc = package_dir + f"{utility}_{map_type}_{active_map.name}"+".png"
        print(f"\nExporting thumbnail to {thumb_loc}...")
        
        map_view.exportToPNG(thumb_loc, width=600, height=400)
        
        # Loop through layers to find the first one with a data source
        for layer in all_layers_list:
            
            # Check if the layer is a data layer (e.g., not a Group Layer)
            if layer.supports("DATASOURCE"):
                
                # We found a valid layer!
                source_layer = layer
                found_valid_layer = True                
                # print(f"Found first valid data layer: {source_layer.name}")
                
                # Get the full extent from the layer's data source ---
                desc = arcpy.Describe(source_layer.dataSource)
                source_extent = desc.extent
                
                # Use the extent by setting the environment ---
                arcpy.env.extent = source_extent
                
                print(f"Successfully set extent to layer: {source_layer.name}")
                print(f"New extent: {arcpy.env.extent}")
                # print(f"Source extent object: {source_extent}")
                
                # Exit the loop 
                break
            
            else:
                # This layer was skipped (e.g., it's a Group Layer)
                print(f"\nSkipping layer (not a data layer): {layer.name}")

        # If the loop finished without finding any valid layers
        if not found_valid_layer:
            print("Error: No layers with a valid data source were found in the map.")
    
    workspace = aprx.defaultGeodatabase
    # print(f"Project's Default Geodatabase: {workspace}")

    # Set the default geodatabase
    if workspace and arcpy.Exists(workspace):
        arcpy.env.workspace = workspace
        print(f"\nWorkspace has been set to: {arcpy.env.workspace}")
        
    else:
        print("\nCould not find a valid geodatabase to set the workspace.")

except Exception as e:
    print(f"Error accessing the project or active map: {e}")
    # exit if no map is active.
    exit()

# Name for the output index feature class that will be created.
output_index_features = f"{utility}_{map_type}_{active_map.name}_Tile_Index"
output_vtpk_package = package_dir + f"{utility}_{map_type}_{active_map.name}"+".vtpk" 


The active map is: Calgary

Successfully set map CS to: WGS_1984_Web_Mercator_Auxiliary_Sphere

Setting scale for: BELLWEST_NETWORK_ABNORTH with 300000
Setting scale for: POLE with 10000
Setting scale for: HUBS with 10000
Setting scale for: MANHOLES with 10000
Setting scale for: PED with 300000
Setting scale for: CO with 50000
Setting scale for: POP with 10000
Setting scale for: VAULT with 10000
Setting scale for: SPLICE with 300000
Setting scale for: HANDHOLES with 10000
Setting scale for: PULL_BOX with 10000
Setting scale for: NATIONAL_FOTS with 10000
Setting scale for: AB_CONDUIT with 300000
Setting scale for: BELL_WEST_CONDUIT with 300000
Setting scale for: COLDLAKEFIBER_AREA with 300000

Skipping Base Maps: World Topographic Map

Skipping Base Maps: World Hillshade

 ----- Scale Updated.-----
The first Layer in TOC is : ['BELLWEST_NETWORK_ABNORTH']

Renaming active map from 'Calgary' to 'ABNORTH'...

Exporting thumbnail to C:\Users\sameer.bajracharya\Sam_works\DEV\ArcGIS_Enterpris

In [34]:
try:   
    # Construct the full path for the output.
    out_features_path = os.path.join(workspace, output_index_features)
    # print(out_features_path)
    
    # --- Record the start time ---
    script_start_time = time.perf_counter()
    print(f"\nStarting to create vector tile index for the map: '{active_map.name}'...")

   # Execute the Create Vector Tile Index tool using the active map as input.
    arcpy.management.CreateVectorTileIndex(
        in_map=active_map,
        # out_featureclass=r"C:\Users\sameer.bajracharya\Sam_works\DEV\ArcGIS_Enterprise\OW01_VTPK_1M_500_test.shp",
        out_featureclass=out_features_path,
        service_type="ONLINE",
        tiling_scheme=None,
        vertex_count=10000,
    )

    print(f"Successfully created index features at: {out_features_path}\n")

    # Remove the tile index layer from the layerlist before creating the vector tile package.
    remove_layer_name = output_index_features

    layer_to_remove = None
    
    for lyr in active_map.listLayers():
        if lyr.name == remove_layer_name:
            layer_to_remove = lyr
            break
    
    if layer_to_remove:
        active_map.removeLayer(layer_to_remove)
        print(f"Successfully removed layer from content: '{remove_layer_name}'\n")
    else:
        print(f"Error: Layer '{remove_layer_name}' not found in the map '{active_map.name}'.")

    # Creating a vector tile package using the index feature 
    print(f"Starting to create vector tile package for the map: '{active_map.name}'...")
    
    arcpy.management.CreateVectorTilePackage(
        in_map=active_map,
        output_file= output_vtpk_package,
        service_type="ONLINE",
        tiling_scheme=None,
        tile_structure="INDEXED",
        min_cached_scale=288895, #1155581, 577791, 288895 or 144448
        max_cached_scale=282, #564 or 282 or 141
        # index_polygons=r"C:\Users\sameer.bajracharya\Sam_works\DEV\ArcGIS_Enterprise\Alektra_HN01_Network_VTPK_Index_Feature.shp",
        index_polygons= out_features_path,
        summary="",
        tags=""
    )

    print(f"Successfully created Vector Tile Package at: {output_vtpk_package}")

except arcpy.ExecuteError:
    # Get and print geoprocessing error messages.
    print(arcpy.GetMessages(2))

except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")

finally:
    # --- 3. This block always runs (success or error) ---
    script_end_time = time.perf_counter()
    elapsed_time = script_end_time - script_start_time
    
    # Optional: Convert to minutes and seconds for readability
    minutes = int(elapsed_time // 60)
    seconds = elapsed_time % 60
    
    print("\n---------------------------------")
    print(f"Script finished in {minutes} minutes and {seconds:.2f} seconds.")
    print(f"(Total elapsed: {elapsed_time:.2f} seconds)")



Starting to create vector tile index for the map: 'ABNORTH'...
Successfully created index features at: C:\Users\sameer.bajracharya\Documents\ArcGIS\Packages\Alectra_w_definition_queries_1a3bd1\commondata\Publishing.gdb\BELLWEST_NETWORK_ABNORTH_Tile_Index

Successfully removed layer from content: 'BELLWEST_NETWORK_ABNORTH_Tile_Index'

Starting to create vector tile package for the map: 'ABNORTH'...
Successfully created Vector Tile Package at: C:\Users\sameer.bajracharya\Sam_works\DEV\ArcGIS_Enterprise\VTPK\VTPK_BELLWEST_NETWORK_ABNORTH.vtpk

---------------------------------
Script finished in 0 minutes and 22.07 seconds.
(Total elapsed: 22.07 seconds)


In [35]:
# This creates a Hosted Vector Tile Layer
from arcgis.gis import GIS

# Connect to your ArcGIS Online or Portal
# In an ArcGIS Notebook, "home" connects automatically
gis = GIS("home")
user = gis.users.me

# Update the name of the folder from ArcGIS Enterprise/online
folder_name = "VTPK"

# Set the item properties for the new item in your portal
item_properties = {
    "title": f"{utility}_{map_type}_{active_map.name}_VTPK",
    "tags": f"Vector, Tiles, {active_map.name}, {map_type}, {utility}",
    "type": "Vector Tile Package",
    "description": f"{utility} {map_type} {active_map.name} Hosted Vector Tile Layers."
}
# print(item_properties)

# Add the .vtpk to your portal as an item
print("Uploading the .vtpk package...")
get_folder = gis.content.folders.get(folder_name)

vtpk_item = get_folder.add(item_properties, 
                            file=output_vtpk_package)

print("Waiting for VTPK item processing...")
while not vtpk_item.done():
    print("...Processing, please wait...")
    time.sleep(5)
    
new_item = vtpk_item.result()
print(f"VTPK uploaded and processed successfully: {new_item.title}")

# # Publish the item to create the hosted layer
print("Publishing hosted vector tile layer...")
hosted_tile_layer_item = new_item.publish()

print(f"\nSuccessfully published layer: {hosted_tile_layer_item.url}")

Uploading the .vtpk package...


C:\Users\sameer.bajracharya\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\concurrent\futures\thread.py:58: ResourceWarning: unclosed file <_io.BufferedReader name='C:\\Users\\sameer.bajracharya\\Sam_works\\DEV\\ArcGIS_Enterprise\\VTPK\\VTPK_BELLWEST_NETWORK_ABNORTH.vtpk'>
  result = self.fn(*self.args, **self.kwargs)


Waiting for VTPK item processing...
VTPK uploaded and processed successfully: BELLWEST_NETWORK_ABNORTH_VTPK
Publishing hosted vector tile layer...

Successfully published layer: https://gis-sandbox.planview.ca/server/rest/services/Hosted/BELLWEST_NETWORK_ABNORTH_VTPK/VectorTileServer


In [36]:
# Set Extent and Update Thumbnail
print("\nSetting Extent and Thumbnail...")

api_extent = [
    source_extent.XMin,
    source_extent.YMin,
    source_extent.XMax,
    source_extent.YMax,
]

# custom extent
# api_extent = [-9331983.83, 5001946.96, -8489798.92, 5763191.07]

print(f"Setting item's bounding box {api_extent}.")

hosted_tile_layer_item.update(
    item_properties={
        "extent": api_extent
    },
    thumbnail=thumb_loc
)
        
print("--- Thumbnail updated successfully! ---")
print(f"Published layer Link: {hosted_tile_layer_item.url}")


Setting Extent and Thumbnail...
Setting item's bounding box [-118.80037799999997, 49.63291501500004, -110.66995997999999, 55.170211995000045].
--- Thumbnail updated successfully! ---
Published layer Link: https://gis-sandbox.planview.ca/server/rest/services/Hosted/BELLWEST_NETWORK_ABNORTH_VTPK/VectorTileServer
